# ModernBERT-JEPA decider: quick check (Head B, JevBench)

The smallest end-to-end run of the design in `specs/ModernBERT-JEPA-Spec-v1.md`, with every component in place:

- **context encoder** (ModernBERT-base, trained) reads the question and the state;
- **question-conditioned pooling** (Head B) attends from the question to the state's tokens;
- **predictor** maps the reading to a predicted answer vector;
- **target encoder** (an EMA copy of the context encoder, no gradients) embeds each option's text;
- **head**: probabilities are a softmax over cosine similarity between the prediction and each option, with a temperature per question type.

Training has two short stages in one run: **Stage 1**, a filtered sample of public typed decisions (`tasksource/tasksource-jev-typed-decisions`: commercial-use licences only, nothing overlapping either benchmark); **Stage 2**, a subset of our generated data. The test is JevBench's 231 public items (`original`, `easy`, `hard`), scored here with JevBench's measures after each stage; they are never trained on.

## Parameters

In [ ]:
import os

def env(name, default):
    return os.environ.get(name, default)

ENCODER = env("ENCODER", "answerdotai/ModernBERT-base")
DATASET = "skamalj/signpost-d-decisions"           # our generated data (private): train and dev
JEVBENCH_COMMIT = "609a664e81f46e03731600bd7bd9f1b7a61257c5"   # pinned public items
TASKSOURCE = "tasksource/tasksource-jev-typed-decisions"   # Stage 1: public typed decisions
TS_QUESTIONS = int(env("TS_QUESTIONS", "20000"))   # Stage 1 sample size
TS_PER_SOURCE = int(env("TS_PER_SOURCE", "300"))   # at most this many from one source
TS_BUFFER = 50000                                  # streaming shuffle buffer
OVERLAP_N = 8                                      # drop a state sharing an 8-word run with a test
TRAIN_QUESTIONS = int(env("TRAIN_QUESTIONS", "6000"))  # Stage 2: a subset of our 12,274
EPOCHS = int(env("EPOCHS", "1"))
BATCH = int(env("BATCH", "16"))                    # questions per step
TRAIN_MAX_LEN = 1024                               # our records are short JSON
EVAL_MAX_LEN = 8192                                # JevBench's hard items are long documents
OPTION_MAX_LEN = 64
EVAL_TOKEN_BUDGET = 16384                          # tokens per evaluation batch
LR_ENCODER = float(env("LR_ENCODER", "3e-5"))      # top layer; lower layers decay by LLRD
LLRD = 0.85
LR_HEAD = float(env("LR_HEAD", "5e-4"))            # pooling, projection, predictor, temperatures
WARMUP = 0.05
EMA_START, EMA_END = 0.996, 0.9999                 # target-encoder momentum, cosine schedule
LAMBDA_JEPA, LAMBDA_ORD, LAMBDA_VC = 0.5, 0.5, 0.04
TAU_INIT = 0.07
SEED = int(env("SEED", "42"))

## Packages and the Hugging Face token

The token reads our private dataset; only where it came from is printed.

In [ ]:
!pip install -q "transformers>=4.48"

import copy, json, math, random, statistics, time, urllib.request
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoTokenizer


def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle secret"
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN"), "Colab secret"
    except Exception:
        return None, "NOT FOUND"


HF_TOKEN, token_from = hf_token()
print(f"Hugging Face token: {token_from}")

## Device and log

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bf16 where the GPU has it; fp32 otherwise (ModernBERT overflows in fp16)
BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()
random.seed(SEED)
torch.manual_seed(SEED)


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


gpu = torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU"
log(f"torch {torch.__version__} | transformers {transformers.__version__} | {gpu} | "
    f"{'bf16' if BF16 else 'fp32'} | {ENCODER}")

## One item per question

Every question becomes the same shape: question text, state text, option names, option texts and a gold distribution over the options. Option texts: `noul` uses the criteria's `true` / `false` descriptions; `choice` is `name: description`; `score` is `Level i of L-1: level`.

In [ ]:
def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def options_of(q):
    """-> (option names, option texts) for one System One question."""
    kind, crit = q["type"], q.get("criteria")
    if kind == "noul":
        crit = crit or {}
        names = ["true", "false"]
        texts = [crit.get("true") or "The statement is true.",
                 crit.get("false") or "The statement is false."]
    elif kind == "choice":
        names = list(crit)
        texts = [f"{n.replace('_', ' ')}: {d}" if d else n.replace("_", " ")
                 for n, d in crit.items()]
    else:
        last = len(crit) - 1
        names = [str(i) for i in range(len(crit))]
        texts = [f"Level {i} of {last}: {level}" for i, level in enumerate(crit)]
    return names, texts


def make_item(kind, question, state, q, gold, source, family=""):
    names, texts = options_of(q)
    total = sum(gold)
    return {"type": kind, "question": question or "", "state": state_text(state), "names": names,
            "texts": texts, "gold": [g / total for g in gold], "source": source, "family": family}

## Our training and dev data

In [ ]:
def load_split(split):
    path = hf_hub_download(DATASET, f"data/{split}.jsonl", repo_type="dataset", token=HF_TOKEN)
    items = []
    for line in open(path, encoding="utf-8"):
        rec = json.loads(line)
        for qid, q in rec["questions"].items():
            probs = rec["gold"][qid]["probabilities"]
            names, _ = options_of(q)
            gold = [float(probs[n]) for n in names]
            items.append(make_item(q["type"], q.get("instructions"), rec["state"], q, gold, split))
    return items


train_all, dev_items = load_split("train"), load_split("dev")
rng = random.Random(SEED)
train_items = rng.sample(train_all, min(TRAIN_QUESTIONS, len(train_all)))
print(f"train {len(train_items)} of {len(train_all)} questions "
      f"{dict(Counter(it['type'] for it in train_items))} | dev {len(dev_items)}")

## JevBench public items (test only, never trained on)

One question per item and a single expected label, so the gold is one-hot.

In [ ]:
def load_jevbench(split):
    url = (f"https://raw.githubusercontent.com/fstandhartinger/jevbench/{JEVBENCH_COMMIT}"
           f"/datasets/public/{split}.jsonl")
    items = []
    for line in urllib.request.urlopen(url).read().decode("utf-8").splitlines():
        rec = json.loads(line)
        q = rec["question"]
        names, _ = options_of(q)
        if q["type"] == "noul":
            target = "true" if rec["expected"] == "yes" else "false"
        else:
            target = str(rec["expected"])
        gold = [1.0 if n == target else 0.0 for n in names]
        assert sum(gold) == 1, rec["id"]
        items.append(make_item(q["type"], q["instructions"], rec["state"], q, gold,
                               split, rec["family"]))
    return items


jev_items = [it for split in ("original", "easy", "hard") for it in load_jevbench(split)]
print(f"JevBench public: {len(jev_items)} items {dict(Counter(it['source'] for it in jev_items))} "
      f"{dict(Counter(it['type'] for it in jev_items))}")

## Stage 1 data: public typed decisions, filtered

A guard set of word 8-grams from every test state we care about (JevBench's public items and Typed Decisions' train and test splits). Then the tasksource stream, shuffled: keep rows whose `license_use` is commercial, with 2–50 options (2–10 levels for `score`), at most `TS_PER_SOURCE` per source, and no 8-gram shared with the guard set.

In [ ]:
import re

import pandas as pd
from datasets import load_dataset

WORD = re.compile(r"\w+")


def ngrams(text, n=OVERLAP_N):
    words = WORD.findall(text.lower())
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


guard = set()
for it in jev_items:
    guard |= ngrams(it["state"])
for split in ("train", "test"):
    path = hf_hub_download("LocalLLaMA/typed-decisions", f"all/{split}-00000-of-00001.parquet",
                           repo_type="dataset")
    for state in pd.read_parquet(path)["state"]:
        guard |= ngrams(state_text(json.loads(state)))
print(f"guard set: {len(guard):,} 8-grams")

In [ ]:
def ts_item(row):
    """A tasksource row -> one item, or None if it does not fit."""
    kind, opts, target = row["kind"], row["options"], row["target"]
    if kind == "noul":
        q, gold = {"type": "noul", "criteria": None}, [target[0], 1 - target[0]]
    elif kind == "choice" and 2 <= len(opts) <= 50 and len(set(opts)) == len(opts):
        q, gold = {"type": "choice", "criteria": {o: None for o in opts}}, list(target)
    elif kind == "score" and 2 <= len(opts) <= 10:
        q, gold = {"type": "score", "criteria": list(opts)}, list(target)
    else:
        return None
    if sum(gold) <= 0 or len(gold) != len(options_of(q)[0]):
        return None
    return make_item(kind, row["question"], row["state"], q, gold, "tasksource", row["source"])


stream = load_dataset(TASKSOURCE, "default", split="train", streaming=True, token=HF_TOKEN)
stream = stream.shuffle(seed=SEED, buffer_size=TS_BUFFER)
ts_items, per_source, dropped = [], Counter(), Counter()
for row in stream:
    if row["license_use"] != "commercial":
        dropped["licence"] += 1
        continue
    if per_source[row["source"]] >= TS_PER_SOURCE:
        continue
    item = ts_item(row)
    if item is None:
        dropped["shape"] += 1
        continue
    if ngrams(item["state"]) & guard:
        dropped["overlap"] += 1
        continue
    ts_items.append(item)
    per_source[row["source"]] += 1
    if len(ts_items) >= TS_QUESTIONS:
        break
print(f"Stage 1: {len(ts_items)} questions from {len(per_source)} sources "
      f"{dict(Counter(it['type'] for it in ts_items))} | dropped {dict(dropped)}")

## Tokens

The context is `[CLS] question [SEP] state [SEP]`, with masks marking which tokens are the question and which the state (Head B's pooling needs both). A state longer than the budget is cut at its end; the count of cut items is reported.

In [ ]:
tok = AutoTokenizer.from_pretrained(ENCODER)
CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id


def ids(text):
    return tok(text, add_special_tokens=False)["input_ids"]


def context_batch(items, max_len):
    """-> ids, attention mask, question mask, state mask (padded), and the count of cut states."""
    rows, cut = [], 0
    for it in items:
        q = ids(it["question"])[:256]
        s = ids(it["state"])
        room = max_len - len(q) - 3
        cut += len(s) > room
        s = s[:room]
        rows.append(([CLS] + q + [SEP] + s + [SEP], len(q), len(s)))
    width = max(len(r[0]) for r in rows)
    input_ids, att, qm, sm = [], [], [], []
    for seq, nq, ns in rows:
        pad = width - len(seq)
        input_ids.append(seq + [PAD] * pad)
        att.append([1] * len(seq) + [0] * pad)
        qm.append([0] + [1] * nq + [0] * (ns + 2 + pad))
        sm.append([0] * (nq + 2) + [1] * ns + [0] * (1 + pad))
    as_t = lambda x: torch.tensor(x, device=DEVICE)
    return as_t(input_ids), as_t(att), as_t(qm), as_t(sm), cut


def option_batch(texts):
    enc = tok(texts, add_special_tokens=True, truncation=True, max_length=OPTION_MAX_LEN,
              padding=True, return_tensors="pt")
    return enc["input_ids"].to(DEVICE), enc["attention_mask"].to(DEVICE)

## The model

Context side (trained): encoder → Head B pooling → projection → predictor, giving ẑ. Target side (no gradients): the EMA copy of the encoder and projection embeds each option text, giving y_k. Scores are cos(ẑ, y_k) / τ_type, plus a learned position bias for `noul` and `score` (none for `choice`).

In [ ]:
TYPES = ["noul", "choice", "score"]


def masked_mean(h, mask):
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1)


class QueryPool(nn.Module):
    """Head B: the question's mean vector attends over the state's tokens."""

    def __init__(self, hidden, heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden, heads, batch_first=True)
        self.out = nn.Linear(2 * hidden, hidden)

    def forward(self, h, qmask, smask):
        query = masked_mean(h, qmask)
        smask = smask.clone()
        smask[smask.sum(1) == 0, 0] = 1                # an empty state attends to [CLS]
        found, _ = self.attn(query.unsqueeze(1), h, h, key_padding_mask=smask == 0)
        return self.out(torch.cat([query, found.squeeze(1)], dim=-1))

The decider: the encoder (embeddings frozen), Head B's pooling, the projection and predictor on the context side, a temperature per type, position biases, and the EMA target encoder and projection.

In [ ]:
class JepaDecider(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(ENCODER, attn_implementation="sdpa")
        hidden = self.encoder.config.hidden_size
        for p in self.encoder.embeddings.parameters():
            p.requires_grad = False
        self.pool = QueryPool(hidden)
        self.proj = nn.Linear(hidden, hidden)
        self.predictor = nn.Sequential(nn.Linear(hidden, 2 * hidden), nn.GELU(),
                                       nn.LayerNorm(2 * hidden), nn.Linear(2 * hidden, hidden))
        self.log_tau = nn.Parameter(torch.full((3,), math.log(TAU_INIT)))
        self.bias_noul = nn.Parameter(torch.zeros(2))
        self.bias_score = nn.Parameter(torch.zeros(10))
        self.target_encoder = copy.deepcopy(self.encoder).requires_grad_(False)
        self.target_proj = copy.deepcopy(self.proj).requires_grad_(False)

    def predict(self, input_ids, att, qm, sm):
        """-> (z: the projected reading, used by the variance term; z_hat: unit prediction)."""
        h = self.encoder(input_ids=input_ids, attention_mask=att).last_hidden_state
        z = self.proj(self.pool(h, qm, sm))
        return z, F.normalize(z + self.predictor(z), dim=-1)

    @torch.no_grad()
    def embed_options(self, texts):
        input_ids, att = option_batch(texts)
        h = self.target_encoder(input_ids=input_ids, attention_mask=att).last_hidden_state
        return F.normalize(self.target_proj(masked_mean(h, att)), dim=-1)

## Scores for a batch

Options of all questions are embedded together, then each question's options are scored and padded into one matrix (missing options get −10⁴).

In [ ]:
def score_batch(model, items, max_len):
    input_ids, att, qm, sm, cut = context_batch(items, max_len)
    z, z_hat = model.predict(input_ids, att, qm, sm)
    texts = [t for it in items for t in it["texts"]]
    y_all = model.embed_options(texts).to(z_hat.dtype)
    width = max(len(it["texts"]) for it in items)
    scores = torch.full((len(items), width), -1e4, device=DEVICE, dtype=torch.float32)
    y_pad = torch.zeros(len(items), width, y_all.shape[-1], device=DEVICE, dtype=torch.float32)
    start = 0
    for i, it in enumerate(items):
        k = len(it["texts"])
        y = y_all[start:start + k].float()
        start += k
        tau = model.log_tau[TYPES.index(it["type"])].exp().clamp(min=0.01)
        s = (y @ z_hat[i].float()) / tau
        if it["type"] == "noul":
            s = s + model.bias_noul
        elif it["type"] == "score":
            s = s + model.bias_score[:k]
        scores[i, :k] = s
        y_pad[i, :k] = y
    return scores, z.float(), z_hat.float(), y_pad, cut

## Losses

KL to the gold, the JEPA latent loss (prediction against the gold-weighted target embedding), the expected-level error for `score`, and a VICReg variance–covariance term against collapse.

In [ ]:
def gold_matrix(items, width):
    g = torch.zeros(len(items), width, device=DEVICE)
    for i, it in enumerate(items):
        g[i, :len(it["gold"])] = torch.tensor(it["gold"], device=DEVICE)
    return g


def losses(scores, z, z_hat, y_pad, items):
    g = gold_matrix(items, scores.shape[1])
    logp = F.log_softmax(scores, dim=1)
    dec = (g * (g.clamp(min=1e-12).log() - logp)).sum(1).mean()
    y_bar = F.normalize((g.unsqueeze(-1) * y_pad).sum(1), dim=-1)
    jepa = (1 - (z_hat * y_bar).sum(1)).mean()
    is_score = torch.tensor([it["type"] == "score" for it in items], device=DEVICE)
    ordinal = torch.zeros((), device=DEVICE)
    if is_score.any():
        levels = torch.arange(scores.shape[1], device=DEVICE, dtype=torch.float32)
        n = torch.tensor([len(it["gold"]) - 1 for it in items], device=DEVICE).clamp(min=1)
        gap = ((logp.exp() - g) * levels).sum(1).abs() / n
        ordinal = gap[is_score].mean()
    zc = z - z.mean(0)
    std = (zc.var(0) + 1e-4).sqrt()
    cov = (zc.T @ zc) / max(len(items) - 1, 1)
    off = cov - torch.diag(torch.diag(cov))
    vc = F.relu(1 - std).mean() + (off ** 2).sum() / z.shape[1]
    total = dec + LAMBDA_JEPA * jepa + LAMBDA_ORD * ordinal + LAMBDA_VC * vc
    return total, {"dec": dec.item(), "jepa": jepa.item(), "ord": ordinal.item(), "vc": vc.item()}

## The measures

JevBench's: accuracy (argmax), multi-class Brier, top-label ECE over 10 equal-width bins (empty bins left out), ordinal MAE for `score`; and KL for our soft dev gold.

In [ ]:
def measure(rows):
    n = len(rows)
    acc = sum(r["ok"] for r in rows) / n
    bins = defaultdict(list)
    for r in rows:
        bins[min(int(r["conf"] * 10), 9)].append(r)
    ece = 0.0
    for b in bins.values():
        right = sum(x["ok"] for x in b) / len(b)
        sure = sum(x["conf"] for x in b) / len(b)
        ece += len(b) / n * abs(right - sure)
    out = {"n": n, "acc": round(acc, 4),
           "brier": round(statistics.mean(r["brier"] for r in rows), 4),
           "ece": round(ece, 4), "kl": round(statistics.mean(r["kl"] for r in rows), 4)}
    mae = [r["mae"] for r in rows if r["mae"] is not None]
    if mae:
        out["ordinal_mae"] = round(statistics.mean(mae), 3)
    return out


def row_of(it, p):
    g = it["gold"]
    top = max(range(len(p)), key=p.__getitem__)
    mae = None
    if it["type"] == "score":
        mae = abs(sum(i * x for i, x in enumerate(p)) - max(range(len(g)), key=g.__getitem__))
    return {"ok": top == max(range(len(g)), key=g.__getitem__), "conf": p[top],
            "brier": sum((a - b) ** 2 for a, b in zip(p, g)),
            "kl": sum(b * math.log(b / max(a, 1e-9)) for a, b in zip(p, g) if b > 0),
            "mae": mae, "type": it["type"], "source": it["source"], "family": it["family"]}

## Evaluation

Batches by token budget (long items alone), probabilities per question, the measures overall and by split, type and family, and the time per item.

In [ ]:
@torch.no_grad()
def evaluate(model, items, max_len, name):
    model.eval()
    order = sorted(range(len(items)), key=lambda i: len(items[i]["state"]))
    rows, cut, start, t0 = [None] * len(items), 0, 0, time.perf_counter()
    while start < len(order):
        batch = [order[start]]
        while start + len(batch) < len(order):
            longest = len(ids(items[order[start + len(batch)]]["state"])) + 300
            if longest * (len(batch) + 1) > EVAL_TOKEN_BUDGET:
                break
            batch.append(order[start + len(batch)])
        group = [items[i] for i in batch]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            scores, _, _, _, c = score_batch(model, group, max_len)
        cut += c
        probs = F.softmax(scores, dim=1).tolist()
        for i, it, p in zip(batch, group, probs):
            rows[i] = row_of(it, p[:len(it["gold"])])
        start += len(batch)
    ms = (time.perf_counter() - t0) / len(items) * 1000
    result = {"all": measure(rows), "ms_per_item": round(ms, 1), "states_cut": cut}
    for key in ("source", "type", "family"):
        values = sorted({r[key] for r in rows})
        result[key] = {v: measure([r for r in rows if r[key] == v]) for v in values}
    log(f"{name}: {result['all']} | {result['ms_per_item']} ms/item | states cut {cut}")
    for key in ("source", "type"):
        log("   " + " | ".join(f"{k} {v['acc']}" for k, v in result[key].items()))
    return result

## Build the model and score it untrained

The untrained model is the floor this run must clearly beat.

In [ ]:
model = JepaDecider().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
total = sum(p.numel() for p in model.parameters()) / 1e6
log(f"model: {trainable:.1f} M trainable of {total:.1f} M (the target encoder is the EMA copy)")
before_jev = evaluate(model, jev_items, EVAL_MAX_LEN, "JevBench public, untrained")
before_dev = evaluate(model, dev_items, TRAIN_MAX_LEN, "our dev, untrained")

## Optimizer, schedule and EMA

Layer-wise learning-rate decay from the top encoder layer; the head parts at a higher rate; warm-up then cosine decay. After every step the target encoder moves towards the context encoder.

In [ ]:
def param_groups(model):
    layers = model.encoder.layers
    groups = []
    for i, layer in enumerate(layers):
        lr = LR_ENCODER * LLRD ** (len(layers) - 1 - i)
        groups.append({"params": [p for p in layer.parameters() if p.requires_grad], "lr": lr})
    in_layers = {id(p) for layer in layers for p in layer.parameters()}
    rest = [p for p in model.encoder.parameters() if p.requires_grad and id(p) not in in_layers]
    groups.append({"params": rest, "lr": LR_ENCODER})
    heads = [model.pool, model.proj, model.predictor]
    head_params = [p for m in heads for p in m.parameters()]
    head_params += [model.log_tau, model.bias_noul, model.bias_score]
    groups.append({"params": head_params, "lr": LR_HEAD})
    return groups


TOTAL_STEPS = EPOCHS * (math.ceil(len(ts_items) / BATCH) + math.ceil(len(train_items) / BATCH))
opt = torch.optim.AdamW(param_groups(model), weight_decay=0.01)
warm = max(1, int(WARMUP * TOTAL_STEPS))


def lr_factor(s):
    if s < warm:
        return (s + 1) / warm
    return 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, TOTAL_STEPS - warm)))


sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)


@torch.no_grad()
def ema_update(model, step):
    m = EMA_END - (EMA_END - EMA_START) * (math.cos(math.pi * step / TOTAL_STEPS) + 1) / 2
    for tgt, src in ((model.target_encoder, model.encoder), (model.target_proj, model.proj)):
        for pt, ps in zip(tgt.parameters(), src.parameters()):
            pt.mul_(m).add_(ps.detach(), alpha=1 - m)
    return m

## Training

One function for both stages; the optimizer, schedule and EMA momentum run across both. Every 50 steps: the losses, the spread of the projected readings (collapse shows as a std near 0), the EMA momentum and the temperatures.

In [ ]:
step = 0


def train_on(items, name):
    global step
    model.train()
    t0 = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        order = list(range(len(items)))
        random.Random(SEED + epoch).shuffle(order)
        for b in range(0, len(order), BATCH):
            batch = [items[i] for i in order[b:b + BATCH]]
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
                scores, z, z_hat, y_pad, _ = score_batch(model, batch, TRAIN_MAX_LEN)
            loss, parts = losses(scores, z, z_hat, y_pad, batch)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            params = [p for g in opt.param_groups for p in g["params"]]
            norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            sched.step()
            step += 1
            m = ema_update(model, step)
            if step % 50 == 0:
                terms = " ".join(f"{k} {v:.4f}" for k, v in parts.items())
                taus = ",".join(f"{t:.3f}" for t in model.log_tau.exp().tolist())
                log(f"{name} step {step}/{TOTAL_STEPS} | loss {loss.item():.4f} {terms} | "
                    f"grad {norm:.2f} | z std {z.std(0).mean().item():.3f} | ema {m:.5f} | "
                    f"tau {taus}")
    log(f"{name} done in {(time.perf_counter() - t0) / 60:.1f} min")

## Stage 1: public typed decisions, then scored

In [ ]:
train_on(ts_items, "stage 1")
mid_jev = evaluate(model, jev_items, EVAL_MAX_LEN, "JevBench public, after stage 1")
mid_dev = evaluate(model, dev_items, TRAIN_MAX_LEN, "our dev, after stage 1")

## Stage 2: our generated data, then scored

In [ ]:
train_on(train_items, "stage 2")
after_dev = evaluate(model, dev_items, TRAIN_MAX_LEN, "our dev, after stage 2")
after_jev = evaluate(model, jev_items, EVAL_MAX_LEN, "JevBench public, after stage 2")

## Side by side

Untrained, after Stage 1 and after Stage 2, on JevBench's splits and on our dev, next to chance (one over the number of options).

In [ ]:
def chance(items):
    return round(statistics.mean(1 / len(it["names"]) for it in items), 3)


print(f"{'set':22s} {'untrained':>10s} {'stage 1':>10s} {'stage 2':>10s} {'chance':>8s}")
for split in ("original", "easy", "hard"):
    sub = [it for it in jev_items if it["source"] == split]
    accs = [r["source"][split]["acc"] for r in (before_jev, mid_jev, after_jev)]
    cols = " ".join(f"{a:>10.3f}" for a in accs)
    print(f"JevBench {split:13s} {cols} {chance(sub):>8.3f}")
for name, runs, sub in (("JevBench all", (before_jev, mid_jev, after_jev), jev_items),
                        ("our dev", (before_dev, mid_dev, after_dev), dev_items)):
    accs = [r["all"]["acc"] for r in runs]
    print(f"{name:22s} " + " ".join(f"{a:>10.3f}" for a in accs) + f" {chance(sub):>8.3f}")
best = after_jev["all"]
print(f"\nJevBench after stage 2: Brier {best['brier']} | ECE {best['ece']} | "
      f"ordinal MAE {best.get('ordinal_mae')} | {after_jev['ms_per_item']} ms/item")
print("by family:", {k: v["acc"] for k, v in after_jev["family"].items()})

## JEPA diagnostics on our dev

How close predictions sit to the gold-weighted option embedding, how spread the readings are, and how distinct the option embeddings are.

In [ ]:
@torch.no_grad()
def diagnostics(model, items):
    model.eval()
    cos_gold, spreads, option_sim = [], [], []
    for b in range(0, len(items), 32):
        group = items[b:b + 32]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            scores, z, z_hat, y_pad, _ = score_batch(model, group, TRAIN_MAX_LEN)
        g = gold_matrix(group, scores.shape[1])
        y_bar = F.normalize((g.unsqueeze(-1) * y_pad).sum(1), dim=-1)
        cos_gold += (z_hat * y_bar).sum(1).tolist()
        spreads.append(z.std(0).mean().item())
        for i, it in enumerate(group):
            y = y_pad[i, :len(it["names"])]
            sim = y @ y.T
            k = len(it["names"])
            option_sim.append(((sim.sum() - k) / (k * (k - 1))).item())
    log(f"cos(prediction, gold target) {statistics.mean(cos_gold):.3f} | "
        f"z std {statistics.mean(spreads):.3f} | "
        f"mean similarity between a question's options {statistics.mean(option_sim):.3f}")


diagnostics(model, dev_items)